### Exploration du dataset et idées pour le modèle de prédiction

### 1. Coûts d'exposition mensuels (/data/costs)
* **EID** : Identifiant unique de l'élément du réseau.
* **MONTH** : Mois au format YYYY-MM (ex. 2022-06).
* **PEAKID** : Profil horaire (0 = OFF / Off-Peak, 1 = ON / On-Peak).
* **C** : Coût d'exposition mensuel associé par le marché.

### 2. Prix réalisés horaires (/data/prices)
* **EID** : Identifiant unique de l'élément du réseau.
* **DATETIME** : Horodatage au format ISO 8601 sans timezone, résolution horaire, suivant la convention Hour Ending (HE).
* **PEAKID** : Profil horaire (0 = OFF, 1 = ON).
* **PRICEREALIZED** : Prix réalisé horaire (les valeurs manquantes équivalent à 0).

### 3. Simulations mensuelles (/data/sim_monthly)
* **SCENARIOID** : Identifiant de scénario (1, 2 ou 3) correspondant à trois méthodes de prédiction différentes.
* **EID** : Identifiant unique de l'élément du réseau.
* **DATETIME** : Horodatage horaire (convention HE).
* **PEAKID** : Profil horaire (0 = OFF, 1 = ON).
* **ACTIVATIONLEVEL** : Intensité de l'opportunité (en %).
* **WINDIMPACT, SOLARIMPACT, HYDROIMPACT, NONRENEWBALIMPACT, EXTERNALIMPACT** : Contributions partielles additives (en %) expliquant une partie de l'intensité (ACTIVATIONLEVEL), dites impacts "par source".
* **LOADIMPACT, TRANSMISSIONOUTAGEIMPACT** : Variables explicatives (en %) associées à l'intensité. Non-additives car elles présentent des chevauchements avec les impacts par source.
* **PSM** : Prix simulé par les simulations mensuelles associé à l'élément pour l'heure considérée.

### 4. Simulations journalières (/data/sim_daily)
* **SCENARIOID** : Identifiant de scénario (1, 2 ou 3).
* **EID** : Identifiant unique de l'élément du réseau.
* **DATETIME** : Horodatage horaire (convention HE).
* **PEAKID** : Profil horaire (0 = OFF, 1 = ON).
* **ACTIVATIONLEVEL** : Intensité de l'opportunité (en %).
* **WINDIMPACT, SOLARIMPACT, HYDROIMPACT, NONRENEWBALIMPACT, EXTERNALIMPACT** : Contributions partielles additives (en %) dites impacts "par source".
* **LOADIMPACT, TRANSMISSIONOUTAGEIMPACT** : Variables explicatives (en %) avec chevauchements (non-additives).
* **PSD** : Prix simulé par les simulations journalières associé à l'élément pour l'heure considérée.

In [2]:
#Charger les libraries nécessaires
import sys
import os
import duckdb
# Add the project root to the Python path
sys.path.append(os.path.abspath('..'))

import pandas as pd
from utilities import customdataloader

In [3]:
# Ajuster ce chemin selon votre setup Google Drive Desktop
DATA_ROOT = r"G:\.shortcut-targets-by-id\1SR7TxhNQjFyS8f2bbuo9pcLvzNMLBMwY\CSD\data"

In [4]:
# Instancier le loader
loader = customdataloader(
    pricefile=os.path.join(DATA_ROOT, 'prices', 'prices.parquet'),
    costfile=os.path.join(DATA_ROOT, 'costs', 'costs.parquet'),
    dsimfile=[os.path.join(DATA_ROOT, 'sim_daily', f'sim_daily_{y}.parquet') for y in [2020,2021,2022,2023]],
    msimfile=[os.path.join(DATA_ROOT, 'sim_monthly', f'sim_monthly_{y}.parquet') for y in [2020,2021,2022,2023]],
)

In [5]:
# Chargement complet de costs + prices (ces fichiers sont petits)
costs = pd.read_parquet(os.path.join(DATA_ROOT, 'costs', 'costs.parquet'))
prices = pd.read_parquet(os.path.join(DATA_ROOT, 'prices', 'prices.parquet'))

print(f"costs:  {costs.shape[0]:,} rows  |  prices: {prices.shape[0]:,} rows")

costs:  9,092 rows  |  prices: 453,167 rows


### Load pour le mois de fevrier 2020

In [6]:
# Explorer un mois spécifique
# Choisir le mois cible et le cutoff correspondant
# Choisir le mois cible et le cutoff correspondant
TARGET_MONTH = '2020-02'       # M+1 : le mois qu'on veut prédire
CUTOFF_DATE  = '2020-01-07'    # 7 de M : moment de décision
 
# Trouver le bon fichier sim_monthly selon l'année cible
target_year = TARGET_MONTH[:4]
sim_file = os.path.join(DATA_ROOT, 'sim_monthly', f'sim_monthly_{target_year}.parquet')
 
# Construire filterdf à partir des EIDs présents dans les simulations du mois cible
sim_eids = duckdb.query(f"""
    SELECT DISTINCT EID
    FROM read_parquet('{sim_file}')
    WHERE strftime(DATETIME, '%Y-%m') = '{TARGET_MONTH}'
""").to_df()
 
known_eids = sim_eids['EID'].values
filterdf = pd.DataFrame({
    'EID': list(known_eids) * 2,
    'MONTH': [TARGET_MONTH] * len(known_eids) * 2,
    'PEAKID': [0] * len(known_eids) + [1] * len(known_eids),
})
print(f"Candidats pour {TARGET_MONTH}: {len(filterdf)} triplets ({len(known_eids)} EIDs uniques)")

Candidats pour 2020-02: 9076 triplets (4538 EIDs uniques)


In [7]:
filterdf.head(5) # Afficher les 5 premiers triplets candidats

,EID,MONTH,PEAKID
0,530,2020-02,0
1,568,2020-02,0
2,580,2020-02,0
3,584,2020-02,0
4,592,2020-02,0


In [8]:
#Simulations mensuelles (pour le mois cible)
monthly_sim = loader.get_monthly_data(filterdf)
print(f"Sim monthly: {monthly_sim.shape}")
monthly_sim.head()

Sim monthly: (9475344, 13)


,SCENARIOID,EID,DATETIME,PEAKID,m_ACTIVATIONLEVEL,m_WINDIMPACT,m_SOLARIMPACT,m_HYDROIMPACT,m_NONRENEWBALIMPACT,m_EXTERNALIMPACT,m_LOADIMPACT,m_TRANSMISSIONOUTAGEIMPACT,m_PSM
0,1,1125,2020-02-01 00:00:00,0,1.634400,-5.353300,0.0,-11.1511,19.6954,-4.8988,-0.9039,3.4335,0.0
1,1,1125,2020-02-01 01:00:00,0,9.476100,13.089800,0.0,11.8470,-17.6423,4.4433,0.5405,-1.4922,0.0
2,1,1125,2020-02-01 02:00:00,0,15.587900,18.202400,0.0,11.4899,-15.2728,3.9731,-0.0317,-1.5181,0.0
3,1,1125,2020-02-01 03:00:00,0,29.146700,27.025299,0.0,11.3364,-10.1010,5.1073,-1.3958,-1.5370,0.0
4,1,1125,2020-02-01 04:00:00,0,33.768299,34.120300,0.0,11.2845,-12.7205,4.9159,-1.0266,-1.3589,0.0


In [11]:
# filterdf pour daily sims = mois M (décision), pas M+1 (cible)
daily_filterdf = pd.DataFrame({
    'EID': list(known_eids) * 2,
    'MONTH': [CUTOFF_DATE[:7]] * len(known_eids) * 2,  # '2020-01' au lieu de '2020-02'
    'PEAKID': [0] * len(known_eids) + [1] * len(known_eids),
})

len(daily_filterdf)

9076

In [13]:
#Simulations journalières (jusqu'au cutoff)
daily_sim = loader.get_daily_data(daily_filterdf, cutoff_date=CUTOFF_DATE)
print(f"Sim daily: {daily_sim.shape}")
daily_sim.head()

Sim daily: (2287152, 13)


,SCENARIOID,EID,DATETIME,PEAKID,d_ACTIVATIONLEVEL,d_WINDIMPACT,d_SOLARIMPACT,d_HYDROIMPACT,d_NONRENEWBALIMPACT,d_EXTERNALIMPACT,d_LOADIMPACT,d_TRANSMISSIONOUTAGEIMPACT,d_PSD
0,1,2,2020-01-01 01:00:00,0,38.148499,12.6173,0.0,5.9827,13.1583,1.6088,1.9061,-5.9611,0.0
1,1,2,2020-01-01 02:00:00,0,36.768700,13.5673,0.0,5.8844,9.6836,2.3517,2.4012,-5.9489,0.0
2,1,2,2020-01-01 03:00:00,0,35.448700,12.4558,0.0,5.8438,9.0488,2.2406,2.9768,-5.8125,0.0
3,1,2,2020-01-01 04:00:00,0,35.409698,13.6630,0.0,5.8284,8.3701,2.0937,2.5709,-5.8844,0.0
4,1,2,2020-01-01 05:00:00,0,36.243500,14.0931,0.0,6.0008,8.9773,2.1207,2.1685,-5.9449,0.0


In [17]:
(daily_filterdf["EID"]==filterdf["EID"]).value_counts()

EID
True    9076
Name: count, dtype: int64

### 1.Comprendre la structure de base du data pour un seul mois, soit le 2020-02

In [20]:
# Combien d'EIDs, heures, scénarios
print(monthly_sim.groupby('SCENARIOID')['EID'].nunique())
print(f"Heures par EID/scénario: {monthly_sim.groupby(['SCENARIOID','EID']).size().describe()}")
print(f"Répartition PEAKID:\n{monthly_sim['PEAKID'].value_counts()}")

SCENARIOID
1    4538
2    4538
3    4538
Name: EID, dtype: int64
Heures par EID/scénario: count    13614.0
mean       696.0
std          0.0
min        696.0
25%        696.0
50%        696.0
75%        696.0
max        696.0
dtype: float64
Répartition PEAKID:
PEAKID
0    5118864
1    4356480
Name: count, dtype: int64


Structure des données monthly_sim

- **3 scénarios** de simulation (SCENARIOID 1, 2, 3), chacun contenant exactement **4 538 EIDs**.
- Chaque combinaison (scénario, EID) contient **696 heures** → correspond aux 29 jours × 24h de février 2020 (année bissextile).
- **Répartition ON/OFF-peak** : plus de lignes OFF-peak (5 118 864) que ON-peak (4 356 480), ce qui reflète le fait que les heures hors-pointe (nuits, weekends) sont plus nombreuses.
- **Total** : 9 475 344 lignes = 3 × 4 538 × 696
- Les `filterdf` et `daily_filterdf` partagent les mêmes EIDs mais visent des mois différents : `filterdf` cible M+1 (simulations mensuelles futures) et `daily_filterdf` cible M (observations journalières passées jusqu'au cutoff).

### 2.Vérifier les valeurs manquantes et les zéros

In [21]:
# PSM semble souvent à 0 — vérifier la sparsité
print(monthly_sim.isnull().sum())
print(f"PSM == 0: {(monthly_sim['m_PSM'] == 0).mean():.2%}")
print(f"ACTIVATIONLEVEL == 0: {(monthly_sim['m_ACTIVATIONLEVEL'] == 0).mean():.2%}")

SCENARIOID                    0
EID                           0
DATETIME                      0
PEAKID                        0
m_ACTIVATIONLEVEL             0
m_WINDIMPACT                  0
m_SOLARIMPACT                 0
m_HYDROIMPACT                 0
m_NONRENEWBALIMPACT           0
m_EXTERNALIMPACT              0
m_LOADIMPACT                  0
m_TRANSMISSIONOUTAGEIMPACT    0
m_PSM                         0
dtype: int64
PSM == 0: 99.72%
ACTIVATIONLEVEL == 0: 3.73%


### 3. Distributions des variables clés

In [22]:
monthly_sim[['m_ACTIVATIONLEVEL','m_WINDIMPACT','m_SOLARIMPACT','m_HYDROIMPACT',
             'm_NONRENEWBALIMPACT','m_EXTERNALIMPACT','m_PSM']].describe()

,m_ACTIVATIONLEVEL,m_WINDIMPACT,m_SOLARIMPACT,m_HYDROIMPACT,m_NONRENEWBALIMPACT,m_EXTERNALIMPACT,m_PSM
count,9.475344e+06,9.475344e+06,9.475344e+06,9.475344e+06,9.475344e+06,9.475344e+06,9.475344e+06
mean,2.527923e+01,1.612429e+01,1.066986e-01,-1.918851e+00,7.558743e+00,1.120506e+00,-1.039692e-01
std,2.105917e+01,1.857158e+04,1.520938e+02,5.276585e+03,1.443928e+04,1.788902e+03,3.441647e+00
min,0.000000e+00,-4.901827e+05,-5.852708e+04,-1.622958e+07,-4.440824e+07,-3.163503e+05,-8.650000e+02
25%,7.822300e+00,0.000000e+00,0.000000e+00,-2.536500e+00,0.000000e+00,-1.050100e+00,0.000000e+00
50%,2.134080e+01,3.710700e+00,0.000000e+00,0.000000e+00,8.492100e+00,0.000000e+00,0.000000e+00
75%,3.821940e+01,1.533260e+01,1.900000e-03,2.069200e+00,2.296470e+01,1.153600e+00,0.000000e+00
max,2.948008e+02,5.715320e+07,4.610355e+05,2.768222e+05,1.203789e+06,5.089868e+06,1.745121e+02


### 4. Comparer les 3 scénarios

In [23]:
# Est-ce que les scénarios divergent beaucoup ?
monthly_sim.groupby('SCENARIOID')[['m_ACTIVATIONLEVEL','m_PSM']].mean()

,m_ACTIVATIONLEVEL,m_PSM
SCENARIOID,,
1,24.791637,-0.072150
2,25.579535,-0.103482
3,25.466507,-0.136275


### 5. Le lien pour simulation vs profitabilité

In [26]:
# Calculer le profit réel pour ce mois cible
target_filter = filterdf.copy()
target_profits = loader.get_price_cost_profit(filterdf=target_filter)

# Agréger les sims au niveau mensuel par EID/PEAKID (moyenne sur les heures et scénarios)
sim_agg = monthly_sim.groupby(['EID','PEAKID']).agg(
    mean_activation=('m_ACTIVATIONLEVEL', 'mean'),
    std_activation=('m_ACTIVATIONLEVEL', 'std'),
    mean_psm=('m_PSM', 'mean'),
    sum_psm=('m_PSM', 'sum'),
).reset_index()

#joindre avec profits
analysis = pd.merge(sim_agg, target_profits, on=['EID','PEAKID'], how='inner')
analysis['profitable'] = analysis['PROFIT'] > 0
print(f"% profitable: {analysis['profitable'].mean():.2%}")

% profitable: 14.53%


In [25]:
# Comparer les distributions profitable vs non-profitable
analysis.groupby('profitable')[['mean_activation','std_activation','mean_psm','sum_psm','COST','PRICE']].mean()

,mean_activation,std_activation,mean_psm,sum_psm,COST,PRICE
profitable,,,,,,
False,44.727924,20.299675,-1.730577,-1805.809692,769.796875,-798.679688
True,43.616867,19.548132,-1.224295,-1299.754150,8.451050,501.793396


Le coût (COST) est le facteur discriminant principal. Les opportunités non-profitables ont un coût d'exposition moyen de ~770, contre seulement ~8.5 pour les profitables. 

Le prix réalisé (PRICE) change de signe entre les deux groupes : positif (+502) pour les profitables, négatif (−799) pour les non-profitables. C'est la double peine : coût élevé et prix défavorable.

Les simulations (PSM, activation) sont faiblement discriminantes à elles seules. L'écart sur mean_psm (−1.22 vs −1.73) va dans le bon sens mais reste modeste. Les simulations seront utiles en combinaison avec d'autres features, mais insuffisantes seules pour séparer les classes.

In [27]:
# Creuser le PSM par scénario
monthly_sim.groupby(['SCENARIOID']).agg(
    mean_psm=('m_PSM','mean'),
    nonzero_psm=('m_PSM', lambda x: (x != 0).mean())
).reset_index()

# Regarder si sum_psm a un seuil discriminant
analysis.groupby('profitable')['sum_psm'].describe()

# Explorer les impacts par source
sim_impacts = monthly_sim.groupby(['EID','PEAKID']).agg(
    wind=('m_WINDIMPACT','mean'),
    solar=('m_SOLARIMPACT','mean'),
    hydro=('m_HYDROIMPACT','mean'),
    nonrenew=('m_NONRENEWBALIMPACT','mean'),
    external=('m_EXTERNALIMPACT','mean'),
).reset_index()

impact_analysis = pd.merge(sim_impacts, target_profits, on=['EID','PEAKID'], how='inner')
impact_analysis['profitable'] = impact_analysis['PROFIT'] > 0
impact_analysis.groupby('profitable')[['wind','solar','hydro','nonrenew','external']].mean()

,wind,solar,hydro,nonrenew,external
profitable,,,,,
False,28.788996,0.099887,-2.301787,19.718618,-0.120071
True,23.231472,0.106570,-0.602976,15.580230,0.294873


Wind et nonrenew sont les deux impacts les plus discriminants. Les opportunités non-profitables présentent un impact éolien (+5.6 pts) et non-renouvelable (+4.1 pts) systématiquement plus élevé. Cela suggère que les éléments du réseau fortement exposés aux variations éoliennes et thermiques sont plus risqués.

L'hydro a un effet protecteur relatif. Un impact hydro moins négatif (−0.6 vs −2.3) caractérise les profitables. L'hydraulique, plus prévisible et pilotable, réduit l'incertitude sur les prix.

Solar est non-informatif pour février 2020. Cohérent avec la saisonnalité —> à vérifier sur les mois d'été.

External est marginalement utile mais l'écart reste faible (0.41 pt).

#-----------------------------------------------------
* Répéter l'analyse sur un mois d'été pour comparer
* Est-ce que solar devient discriminant en juillet ?
* Est-ce que le pattern hydro persiste ?

#----------------------------------------------------